[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anicka-net/nla-at-home/blob/main/notebooks/01_read_a_mind.ipynb)

# 01 · Read a Mind

### HAAISS workshop — hands-on part 1 of 3

We take an ordinary open model (**Qwen 2.5 7B**), let it answer a question, and then — instead of reading its *words* — we read the **activation vector** inside layer 20 and ask a second network to **describe that vector in English**.

That second network is an **NLA** (Natural Language Autoencoder): a small LoRA adapter trained to turn a model's internal state into a caption. Think of it as a subtitle track for thought.

**Setup:** `Runtime → Change runtime type → T4 GPU`, then run every cell top to bottom. First run downloads ~5 GB and takes a few minutes.

## Install

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes

## Configure

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE       = "Qwen/Qwen2.5-7B-Instruct"          # the model whose mind we read
AV_ADAPTER = "anicka/nla-qwen2.5-7b-universal-av-grpo"  # universal verbalizer: ONE adapter, all 28 layers (GRPO-refined)
LAYER      = 20   # default readout layer — the universal adapter serves ALL of 0..27; try others
DEPTH_PCT  = 71                                   # <-- NOT cosmetic. The adapter was TRAINED
                                                  #     at 71% depth (layer 20 of 28). This
                                                  #     number is a CONDITIONING INPUT to the
                                                  #     verbalizer. Notebook 02 lets you feel
                                                  #     what happens when you lie about it.
INJECT_CHAR  = "\u320e"                          # the placeholder token we overwrite: ㈎
INJECT_SCALE = 150.0                              # we normalize the activation's L2 norm TO this

## Load the model (once)

We load Qwen in 4-bit and attach the **AV adapter** (`av` = *activation → verbalization*). The base model is frozen; the adapter is 80 MB.

In [ ]:
device = "cuda"
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"

# 4-bit so a 7B model + adapters fit a free-Colab T4 (16 GB). fp16 compute:
# the GRPO-sharpened adapter is numerically sensitive, and fp16 on CUDA is a
# tested-safe path (bf16 on Apple MPS collapses it; not our case here).
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)

tok  = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                            device_map={"": 0})
model = PeftModel.from_pretrained(base, AV_ADAPTER).eval()   # adapter name = "default"

inject_id = tok.encode(INJECT_CHAR, add_special_tokens=False)
assert len(inject_id) == 1, f"injection char must be ONE token, got {inject_id}"
inject_id = inject_id[0]
print("loaded — base + AV adapter on", next(model.parameters()).device)

## The two moves
`read_activation(prompt)` runs the model and grabs the layer-20 vector.
`describe(vector)` injects that vector into the verbalizer and reads out a caption.

In [ ]:
def get_layers(m):
    """Reach the transformer block list through the PEFT + CausalLM wrappers."""
    b = m.base_model.model if hasattr(m, "base_model") else m
    inner = b.model if hasattr(b, "model") else b
    return inner.layers

def read_activation(prompt, layer=LAYER, max_new_tokens=128):
    """Run the model on `prompt` and grab the residual-stream vector at `layer`,
    at the last prompt token (the position that decides the next word)."""
    chat = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    inp = tok(chat, return_tensors="pt").to(device)

    grab = {}
    def hook(mod, inpt, out):
        h = out[0] if isinstance(out, tuple) else out
        if "h" not in grab:                 # FIRST forward pass only — otherwise
            grab["h"] = h[:, -1, :].detach() # every generated token overwrites it
    handle = get_layers(model)[layer].register_forward_hook(hook)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    handle.remove()
    reply = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
    return grab["h"].squeeze(0), reply

def normalize_to(v, scale=INJECT_SCALE):
    """Rescale v so its L2 norm equals `scale`. NOT v * scale — see notebook 02."""
    n = v.float().norm().clamp_min(1e-12)
    return v * (scale / n)

def av_prompt(depth_pct):
    return (
        "You are a meticulous AI researcher conducting an important investigation "
        "into activation vectors from a language model. Your overall task is to "
        "describe the semantic content of that activation vector.\n\n"
        "We will pass the vector enclosed in <concept> tags into your context, "
        "along with the network depth where it was extracted. "
        "You must then produce an explanation for the vector, enclosed within "
        "<explanation> tags. The explanation consists of 2-3 text snippets "
        "describing that vector.\n\n"
        f"Here is the vector from depth {depth_pct}% of the network:\n\n"
        f"<concept>{INJECT_CHAR}</concept>\n\n"
        "Please provide an explanation.\n\n"
        "<explanation>")

def describe(activation, depth=DEPTH_PCT, max_new_tokens=120, scale_fn=normalize_to, **gen_kw):
    """The whole NLA read: build the prompt, overwrite the placeholder token's
    embedding with the (rescaled) activation, let the model narrate."""
    chat = tok.apply_chat_template([{"role": "user", "content": av_prompt(depth)}],
                                   tokenize=False, add_generation_prompt=True)
    ids = tok.encode(chat, add_special_tokens=False)  # match training: chat-wrapped, no BOS
    pos = ids.index(inject_id)
    emb = model.get_input_embeddings()(torch.tensor([ids], device=device)).clone()
    emb[0, pos, :] = scale_fn(activation.to(emb.dtype))
    gen_args = dict(do_sample=False)
    gen_args.update(gen_kw)                    # e.g. do_sample=True, temperature=0.8
    with torch.no_grad():
        out = model.generate(inputs_embeds=emb, max_new_tokens=max_new_tokens,
                             pad_token_id=tok.eos_token_id, **gen_args)
    seq = out[0]
    gen = seq[len(ids):] if seq.shape[0] > len(ids) else seq  # embeds path returns new-only
    return tok.decode(gen, skip_special_tokens=True).split("</explanation>")[0].strip()

## Try it

Change the prompt. The **OUTPUT** is what Qwen would normally say. The **LAYER-20 READOUT** is what the NLA sees happening *inside* the model at 71% depth — often the topic/structure it has committed to, before the words come out.

In [ ]:
# === CHANGE THIS ===
prompt = "Explain how a hash map handles collisions."

activation, reply = read_activation(prompt)
readout = describe(activation)

print("PROMPT :", prompt)
print("\nOUTPUT (what Qwen says):\n ", reply[:400])
print("\nLAYER-20 READOUT (what the NLA sees inside):\n ", readout)

## Now make it interesting

Try prompts where the *inside* and the *outside* might differ:

- `"Do you have feelings?"` — does the readout mention self-reference / refusal framing before the model hedges out loud?

- `"Translate 'good morning' into French."` — does layer 20 already hold *French* / *translation task*?

- A half sentence: `"The capital of Australia is"` — the answer is committed inside long before the token appears.

Run several and eyeball whether the caption tracks the *content* or just the *surface form*. (This eyeball test is the real evaluation — numbers hide template hallucination.)

In [ ]:
for p in ["Do you have feelings?",
          "The capital of Australia is",
          "Write a haiku about rain."]:
    act, rep = read_activation(p)
    print("::", p)
    print("   readout:", describe(act))
    print()

## Where do confabulated entities come from?

If the hash-map readout above named a programming language the prompt never
mentioned (C# is a favorite), you caught the last remaining failure mode
red-handed: **entity slot-filling**. The training corpus and the training
descriptions contain no occurrence of "hash" or "C#" *at all* — that clause is
Qwen's own language prior completing a template slot the activation leaves
unspecified. The activation pins the gist ("how-does-X-work question about
hashing, expository answer coming"); where it is silent, the base model speaks.

Two experiments that separate *read* from *filled-in*:

1. **Pin the entity in the prompt.** Entities genuinely written into the
   activation transfer to the readout.
2. **Re-sample the readout.** The gist survives every sample; an unpinned
   entity flips between C#/Java/Python. *What is stable is read; what flips
   is filled in.*

In [ ]:
# 1) entity pinned in the prompt — should appear in the readout
act_java, _ = read_activation("Explain how a Java HashMap handles collisions.")
print("JAVA-PINNED READOUT:\n ", describe(act_java), "\n")

# 2) entity unpinned — re-sample and watch which parts move
act, _ = read_activation("Explain how a hash map handles collisions.")
for t in range(3):
    print(f"sample {t}:\n ", describe(act, do_sample=True, temperature=0.8), "\n")

## Brain in the jar — the whole depth ladder

So far we read ONE depth. But the universal adapter was trained at
thirteen — and the interesting experiment is reading the *same moment of
thought* at every depth and watching WHERE the truth appears.

Spoiler from our test run, so you know what you're looking at: it is NOT
a smooth "tokens → situation → plan" gradient. The early and middle
rungs confabulate — the verbalizer meets an activation that carries only
vague features ("narrative", "kitchen-ish", "quantity tension") and its
decoder prior fills in a complete, internally consistent, WRONG scene: a
cinnamon pie at 4%, a hamburger with no ketchup at 32%, a bar joke at
63%. Around 71-80% the readout suddenly locks onto the actual prompt.
The ladder is a confabulation→content gradient, and you get to find the
rung where it flips.

One forward captures all depths at once (hooks are free); each rung then
costs one `describe()` (~10 s on a T4). Seven rungs below; switch to
`DEPTH_LAYERS_FULL` for all thirteen.

In [ ]:
# trained depth grid (pct of 28 layers -> block index), from the repo's nla_lib
DEPTH_LAYERS_FULL = {4: 1, 10: 3, 17: 5, 25: 7, 32: 9, 40: 11, 47: 13,
                     55: 15, 63: 18, 71: 20, 80: 22, 90: 25, 96: 27}
DEPTH_LAYERS = {p: DEPTH_LAYERS_FULL[p] for p in (4, 17, 32, 47, 63, 80, 96)}

prompt = "The recipe calls for two eggs, but she only has one, so"

chat = tok.apply_chat_template([{"role": "user", "content": prompt}],
                               tokenize=False, add_generation_prompt=True)
inp = tok(chat, return_tensors="pt").to(device)

grabs, handles = {}, []
def _mk(pct):
    def hook(mod, i, o):
        h = o[0] if isinstance(o, tuple) else o
        if pct not in grabs:
            grabs[pct] = h[:, -1, :].detach().squeeze(0)
    return hook
for pct, L in DEPTH_LAYERS.items():
    handles.append(get_layers(model)[L].register_forward_hook(_mk(pct)))
with torch.no_grad():
    model(**inp)                    # ONE forward pass captures every rung
for h_ in handles:
    h_.remove()

for pct in sorted(grabs):
    print(f"\n{'='*22} depth {pct}% (layer {DEPTH_LAYERS[pct]}) {'='*22}")
    print(describe(grabs[pct], depth=pct, max_new_tokens=90))

**The one-question game: which rung first mentions the eggs?** In our
run it was 80% — everything below it was fluent fiction, everything at
it was the real prompt (constraint, substitution strategy, even the
"one egg vs recipe calls for" tension). Then 96% drifts again (the
final block is output plumbing — its residual is halfway out of the
representation basis, and readouts there go strange).

Three lessons packed in one ladder:

1. **A fluent readout is not a faithful readout** — the 32% hamburger
   story was perfectly coherent and entirely invented. Same lesson as
   the entity demo above, now as a function of depth.
2. **Verbalizable content concentrates in a band** (~47-80% here) —
   the same band where notebook 05's Jacobian lens ignites. Two
   completely different instruments, one boundary.
3. **Depth conditioning matters**: the adapter KNOWS 4% activations
   carry little — and instead of saying "nothing here", it improvises.
   Reading the gaps honestly is exactly what separates an
   interpretability tool from a storyteller, and current NLAs are still
   part storyteller at shallow depths. Now you know — and you can
   measure — where.

---
### ✅ Self-check
Expected anchors:
- the model loads without OOM on a **free T4** (4-bit uses ~5–6 GB),
- for `"Explain how a hash map..."` the readout is **coherent English bullets about data structures / hashing / lookup**, not `SpongeBob` / `Bahamas` / random nouns. Garbage means the injection scale is wrong — see notebook 02.

In [ ]:
act, _ = read_activation("Explain how a hash map handles collisions.")
out = describe(act)
print(out)
assert len(out) > 10, "empty readout — check the injection token / adapter load"
print("\nself-check: readout non-empty ✓  (now eyeball that it is on-topic)")